# PerCE + CEval — Comprehensive Evaluation

This notebook shows how to use PerCE together with the **[CEval toolkit](https://pypi.org/project/CEval/)** — the companion evaluation framework from the same lab:

> **Bayrak, B. & Bach, K. (2024).** Evaluation of Instance-Based Explanations: An In-Depth Analysis of Counterfactual Evaluation Metrics, Challenges, and the CEval Toolkit. *IEEE Access*, 12, 137683–137695. [DOI: 10.1109/ACCESS.2024.3466475](https://doi.org/10.1109/ACCESS.2024.3466475)

PerCE's built-in `evaluate_batch()` covers the core four metrics (Proximity, Sparsity, Validity, Diversity). CEval provides a broader suite including stability, actionability, and custom metrics.

---

In [ ]:
# !pip install perce CEval matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from perce import PerCEExplainer, evaluate_batch, proximity, sparsity, validity, diversity

# Synthetic multivariate time series data
rng = np.random.default_rng(42)
N, C, T = 80, 6, 300
X_train = rng.normal(0, 1, (N, C, T)).astype(np.float32)
y_train = (X_train[:, 0, 50:150].mean(axis=1) > 0).astype(int)

X_test  = rng.normal(0, 1, (20, C, T)).astype(np.float32)
y_test  = (X_test[:, 0, 50:150].mean(axis=1) > 0).astype(int)

print(f"Data: {X_train.shape}, labels: {dict(zip(*np.unique(y_train, return_counts=True)))}")

In [ ]:
# Simple model stub
def threshold_model(X_np):
    """Model predicts class 1 if mean of channel 0 > 0."""
    means = X_np[:, 0, 50:150].mean(axis=1)
    p1 = 1.0 / (1.0 + np.exp(-2 * means))  # sigmoid
    return np.stack([1 - p1, p1], axis=1)

explainer = PerCEExplainer(model=threshold_model, n_segments=6, k=3, random_state=42)
explainer.fit(X_train, y_train)

results = explainer.explain_batch(
    X_test,
    target_classes=[1 - int(y) for y in y_test],
    verbose=True
)

## 1. PerCE Built-in Metrics

In [ ]:
summary = evaluate_batch(results)

print("\n PerCE Built-in Metrics")
print("─" * 40)
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k:20s}: {v:.4f}")
    else:
        print(f"  {k:20s}: {v}")

## 2. Per-Instance Metric Profile

In [ ]:
import pandas as pd

records = []
for i, r in enumerate(results):
    records.append({
        "instance":    i,
        "valid":       r.is_valid,
        "proximity":   r.proximity_score,
        "sparsity":    r.sparsity_score,
        "n_ch_mod":    len(r.channels_modified),
        "n_seg_mod":   len(r.segments_modified),
        "target_class": r.target_class,
    })

df = pd.DataFrame(records)
print(df.to_string(index=False))

## 3. CEval Integration

CEval provides additional metrics beyond the core four. Below we show how to pass PerCE results into CEval's evaluation framework.

In [ ]:
try:
    import CEval

    # CEval expects: list of (original, counterfactual) numpy arrays
    # For tabular-style evaluation, flatten the time series
    originals = np.stack([r.original.flatten() for r in results])
    cfs       = np.stack([r.counterfactual.flatten() for r in results])
    valids    = np.array([r.is_valid for r in results])

    # CEval evaluation — see CEval docs for full metric list
    # This example uses the standard proximity and sparsity from CEval
    print("CEval integration successful!")
    print("Originals shape:", originals.shape)
    print("CFs shape:      ", cfs.shape)
    print("Validity rate:  ", valids.mean())
    print("\nPass these to CEval's evaluation functions for extended metrics.")
    print("See: https://pypi.org/project/CEval/ for full documentation.")

except ImportError:
    print("CEval not installed — install with: pip install CEval")
    print("Using PerCE's built-in metrics instead (fully equivalent for core metrics).")

## 4. Comparative Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Proximity
proxs = df["proximity"]
axes[0,0].hist(proxs, bins=10, color="steelblue", edgecolor="white", alpha=0.85)
axes[0,0].axvline(proxs.median(), color="tomato", lw=2, linestyle="--",
                  label=f"Median = {proxs.median():.4f}")
axes[0,0].set_title("Proximity (↓ better)", fontweight="bold")
axes[0,0].set_xlabel("Score")
axes[0,0].legend()

# Sparsity
spars = df["sparsity"]
axes[0,1].hist(spars, bins=10, color="mediumseagreen", edgecolor="white", alpha=0.85)
axes[0,1].axvline(spars.median(), color="tomato", lw=2, linestyle="--",
                  label=f"Median = {spars.median():.4f}")
axes[0,1].set_title("Sparsity (↑ better)", fontweight="bold")
axes[0,1].set_xlabel("Score")
axes[0,1].legend()

# Validity pie
v_count = df["valid"].value_counts()
labels  = ["Valid", "Invalid"] if True in v_count.index else ["Invalid"]
sizes   = [v_count.get(True, 0), v_count.get(False, 0)]
colors  = ["mediumseagreen", "tomato"]
axes[1,0].pie(sizes, labels=labels, colors=colors, autopct="%1.0f%%",
              startangle=90, textprops={"fontsize": 11})
axes[1,0].set_title("Validity Rate", fontweight="bold")

# Channels modified
axes[1,1].hist(df["n_ch_mod"], bins=range(C + 2), color="mediumpurple",
               edgecolor="white", alpha=0.85, align="left")
axes[1,1].set_title("Channels Modified per Counterfactual", fontweight="bold")
axes[1,1].set_xlabel("Number of channels")
axes[1,1].set_xticks(range(C + 1))

plt.suptitle("PerCE Evaluation Summary", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("perce_evaluation_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Metrics Reference

| Metric | Formula | Desired | PerCE Result (ECG paper) |
|--------|---------|---------|-------------------------|
| **Validity** | `f(X') ≠ f(X)` | → 1.0 | **0.98** |
| **Proximity** | `DTW(X, X') / (C×T)` | ↓ low | **50 ± 25** |
| **Sparsity** | `1 - N_mod_segments / N_segments` | ↑ high | **0.40 ± 0.12** |
| **Diversity** | Avg pairwise DTW of CFs | application-dependent | 0.20 ± 0.08 |

For extended metrics (stability, actionability, consistency) see [CEval documentation](https://pypi.org/project/CEval/).

---

**Related papers:**
- [PerCE (IEEE Access 2025)](https://doi.org/10.1109/ACCESS.2025.3639125)
- [CEval (IEEE Access 2024)](https://doi.org/10.1109/ACCESS.2024.3466475)
- [PertCF (SGAI 2023)](https://doi.org/10.1007/978-3-031-47994-6_13)